In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Explorar ventas

In [2]:
# Cargar datos de ventas
path_ventas = "datos/Ventas por Cliente/ventas_con_descuento_aplicado.csv"
ventas = pd.read_csv(path_ventas)

print("Ventas - Filas:", ventas.shape[0])
display(ventas.head())

/var/folders/dw/9r647bwd1kl_g5lxtz0kh4100000gp/T/ipykernel_64855/2649053988.py:3: DtypeWarning: Columns (0: dscto_volumen, 1: ids_descuento_volumen) have mixed types. Specify dtype option on import or set low_memory=False.
  ventas = pd.read_csv(path_ventas)


Ventas - Filas: 2157646


,cod_cliente,cod_canal_comercial,cod_consolidado,zona,distrito,fecha_factura,cod_sku,nombre_sku,monto_real,kilo_real,...,precio_lista_por_unidad,descuento_pct_observado,descuento_pct_matcheado,diferencia_descuento_pct,descuento_aplicado,id_descuento_aplicado,calza_con_descuento,gap,monto_a_precio_lista,costo_descuento
0,1227986,HR,37,SANTIAGO,SANTIAGO CENTRO COSTA,2026-02-16,3112,SALAME PIEZA 1KG LP,8274,1.0,...,8274.0,0.000000e+00,0.0,0.000000e+00,ninguno,NaN,False,False,8274.0,0.0
1,1176438,HR,37,NORTE 1,ANTOFAGASTA,2026-02-04,8512,"SALCHICHON CERVEZA WINTER 1,8 KG",108216,27.0,...,7214.4,-1.110223e-14,0.0,-1.110223e-14,ninguno,NaN,False,False,108216.0,0.0
2,1035323,CB,54,NORTE 2,SAN FELIPE,2026-02-16,3112,SALAME PIEZA 1KG LP,7033,1.0,...,8274.0,-1.499879e+01,0.0,-1.499879e+01,ninguno,NaN,False,True,8274.0,1241.0
3,1011205,CB,32,SUR 1,RANCAGUA,2026-02-16,3112,SALAME PIEZA 1KG LP,7033,1.0,...,8274.0,-1.499879e+01,0.0,-1.499879e+01,ninguno,NaN,False,True,8274.0,1241.0
4,1035323,CB,54,NORTE 2,SAN FELIPE,2026-02-09,8517,SALCHICHON CERVEZA MINI PZA 8x400 GR WI,22308,12.0,...,1486.8,-4.998655e+01,0.0,-4.998655e+01,liquidacion_forzado,NaN,False,True,44604.0,22296.0


In [4]:
canales_relevantes = [
    32,  # COBERTURA
    54,  # VOLUMEN COBERTURA
    # 67,  # MAYORISTAS CADENAS
    # 57,  # MAYORISTA B VOLUMEN
    # 38,  # OTROS MAYORISTAS
    # 55,  # HORECA VOLUMEN
    # 37,  # OTROS HORECA
]

ventas["fecha_factura"] = pd.to_datetime(ventas["fecha_factura"])

FECHA_FIN_DATOS_VENTAS = pd.Timestamp("2026-03-31")

ventas_relevantes = ventas[
    (ventas["cod_consolidado"].isin(canales_relevantes)) &
    (ventas["fecha_factura"] <= FECHA_FIN_DATOS_VENTAS)
]

In [5]:
SKU = 3248
FECHA_INICIO_DSCTO = pd.Timestamp("2025-04-29")
FECHA_FIN_DSCTO = pd.Timestamp("2025-05-04")

ventas_periodo_dscto = ventas_relevantes[:]
    # (FECHA_INICIO_DSCTO <= ventas_relevantes["fecha_factura"]) &
    # (ventas_relevantes["fecha_factura"] <= FECHA_FIN_DSCTO)
# ]
ventas_periodo_sku = ventas_periodo_dscto[
    ventas_periodo_dscto["cod_sku"] == SKU
]
ventas_periodo_sku.head()

,cod_cliente,cod_canal_comercial,cod_consolidado,zona,distrito,fecha_factura,cod_sku,nombre_sku,monto_real,kilo_real,...,precio_lista_por_unidad,descuento_pct_observado,descuento_pct_matcheado,diferencia_descuento_pct,descuento_aplicado,id_descuento_aplicado,calza_con_descuento,gap,monto_a_precio_lista,costo_descuento
25998,1029946,CB,54,SUR 2,CONCEPCION SUR,2026-01-02,3248,MORTADELA LISA LP,10533,3.0,...,10860.0,-3.01105,-3.0,-0.01105,base,1494.0,True,False,10860.0,327.0
26004,1075504,CB,54,SUR 1,TALCA,2026-01-02,3248,MORTADELA LISA LP,19548,6.0,...,10860.0,-10.00000,-10.0,0.00000,volumen_1,75253.0,True,False,21720.0,2172.0
26016,1088043,CB,54,SUR 2,CONCEPCION SUR,2026-01-02,3248,MORTADELA LISA LP,10533,3.0,...,10860.0,-3.01105,-3.0,-0.01105,base,1494.0,True,False,10860.0,327.0
26033,1091853,CB,54,SUR 3,PUERTO MONTT,2026-01-02,3248,MORTADELA LISA LP,10533,3.0,...,10860.0,-3.01105,-3.0,-0.01105,base,1494.0,True,False,10860.0,327.0
26044,1114635,CB,54,SANTIAGO,SANTIAGO CENTRO COSTA,2026-01-02,3248,MORTADELA LISA LP,10533,3.0,...,10860.0,-3.01105,-3.0,-0.01105,base,1494.0,True,False,10860.0,327.0


# Explorar Descuentos

In [ ]:
path_descuentos = "datos/Descuentos históricos/consolidado_descuentos.csv"
descuentos = pd.read_csv(path_descuentos)
print("Descuentos - Filas:", descuentos.shape[0])
display(descuentos.head())

In [ ]:
descuentos["fecha_inicio"] = pd.to_datetime(descuentos["fecha_inicio"])
descuentos["fecha_fin"] = pd.to_datetime(descuentos["fecha_fin"])

In [ ]:
descuentos = descuentos[
    descuentos["cod_consolidado"].isin(canales_relevantes) &
    (descuentos["monto_descuento"] < 0) &
    (descuentos["fecha_fin"] >= pd.Timestamp("2025-01-01")) &
    (descuentos["fecha_inicio"] <= pd.Timestamp("2026-03-31"))
]

In [ ]:
dsctos_base = descuentos[descuentos["tipo_descuento"]=="base"]

In [ ]:
dsctos_base